# BART Bay Area — Saatlik Yolcu Talebi Tahmini

**Problem:** San Francisco Bay Area BART metro ağında, herhangi iki istasyon arasında
**saatlik yolcu sayısını (Throughput)** tahmin eden bir model kurmak ve veriyi iş
sorularıyla analiz etmek.

**Veri:** 2016 + 2017 saatlik origin→destination yolculuk kayıtları (~13.3M satır,
46 istasyon) + istasyon koordinatları. Kaynak: Kaggle BART Ridership.

**Yaklaşım:** Modüler `src/` kodu (yerelde doğrulanmış) GitHub'dan `clone` ile gelir;
bu notebook Kaggle'da tam veriyle çalıştırılır. Akış: **kurulum → veri → EDA (6 iş
sorusu) → feature engineering → temporal eğitim → değerlendirme → sonuç**.

> Bu notebook bir kez **Save & Run All (Commit)** ile çalıştırılır; yayınlanan sürüm
> tüm çıktıları (grafikler, metrikler) taşır — açan kişi yeniden çalıştırmak zorunda değil.

## Kurulum — repo clone + import yolu

Notebook ayarları (sağ panel): **Internet → On** (clone için zorunlu),
**Accelerator → GPU** (model eğitiminde gereklidir; veri yükleme ve EDA için gerekmez).

In [ ]:
import shutil, os, sys

REPO = "bay-area-transit-ridership-forecasting"
URL  = "https://github.com/osman-ozcanli/bay-area-transit-ridership-forecasting.git"

# 1) Her zaman taze kod: eskiyi sil, yeniden clone et
shutil.rmtree(REPO, ignore_errors=True)
os.system(f"git clone -q {URL}")

# 2) Import yolunu ekle
repo_path = f"/kaggle/working/{REPO}"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 3) Eski cache'lenmis src modullerini temizle -> kernel restart gerekmeden taze import
for _m in list(sys.modules):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]

# 4) Self-check: clone gercekten geldi mi? (Internet kapali / push yapilmadiysa uyarir)
train_py = os.path.join(REPO, "src", "models", "train.py")
assert os.path.exists(train_py), "CLONE BASARISIZ -> Internet acik mi? GitHub push yapildi mi?"
print("setup ok. src/models ->", os.listdir(os.path.join(REPO, "src", "models")))

## Veri Yükleme (tam veri)

Kaggle'da `use_sample=False` ile **tam ~13.3M satır** yüklenir (yereldeki 1.12M'lik
örnek değil). `/kaggle/input/bart-ridership/` altında 3 CSV ekli olmalı.

`src/data/load.py` içindeki `load_dataset`: 2016+2017 birleştir → DateTime parse →
istasyon merge → WSPR ekle.

In [ ]:
from src.config import load_config
from src.data.load import load_dataset

cfg = load_config()
cfg["environment"] = "kaggle"   # /kaggle/input path'leri
cfg["use_sample"]  = False      # tam veri (~13.3M)

df = load_dataset(cfg)
print("shape:", df.shape)
print("date range:", df["DateTime"].min(), "->", df["DateTime"].max())
df.head(3)

## Keşifsel Veri Analizi (EDA): 6 İş Sorusu

Modelden önce veriyi **iş sorularıyla** anlıyoruz. Tüm mantık `src/eda.py`'de
(modüler, type-hint + docstring); her soru **kod → grafik → yorum** olarak işlenir.

1. En yoğun istasyon  2. En az popüler rota  3. En yoğun gün
4. Gece (LateNight) yolcuları  5. En popüler rotalar
6. Berkeley → SF için koltuk bulmaya en iyi saat

> EDA **ham** ridership üzerinde çalışır (self-trip dahil — betimsel). Modelleme
> aşamasında (aşağıda) self-trip'ler düşürülür. EDA, `df` build_features'tan **önce** çalışır.

In [ ]:
from src.eda import run_eda

# run_eda: ic kismda zaman + koordinat feature'lari ekler, 6 soruyu hesaplar,
# grafikleri figures_dir'e yazar ve YORUM dondurur. df burada HAM yuklenen veri.
eda = run_eda(df, cfg)

In [ ]:
from IPython.display import Image, display
import os
from src.config import get_paths

figs = str(get_paths(cfg)["figures_dir"])
for _name in ["eda_busiest_stations", "eda_least_routes", "eda_busiest_day",
              "eda_period_hour", "eda_hour_day_heatmap", "eda_popular_routes",
              "eda_berkeley_sf_hourly"]:
    _p = os.path.join(figs, _name + ".png")
    if os.path.exists(_p):
        display(Image(filename=_p))

### EDA Bulguları (yorum)

> Aşağıdaki bulgular **tam veri (~13.3M satır, Kaggle)** üzerinde doğrulanmıştır.

- **En yoğun istasyon:** **EMBR / MONT / POWL** ekseni (toplam yük ~**34M / 33M / 26M**)
  — SF finans bölgesi iş merkezleri; sabah varış / akşam çıkış zirvesini bu istasyonlar taşıyor.
- **En az popüler rota:** **WSPR → SBRN** (ve genel olarak **WSPR** uçlu rotalar) —
  ağın güney ucu, çok düşük talep (manuel eklediğimiz edge-case istasyon).
- **En yoğun gün:** **Çarşamba** (hafta ortası iş günü); Cumartesi–Pazar belirgin düşük.
- **Gece (LateNight):** Toplam yolculuğun ~**%1.2**'si — talep neredeyse tamamen gündüz
  zirve saatlerinde.
- **En popüler rotalar:** İki tip **karışımı** — SF çekirdeği **kısa** koridorları
  (POWL ↔ BALB / MONT / 24TH) **ve** uzun **banliyö → merkez** işe-gidiş rotaları
  (**DUBL / FRMT / WOAK → EMBR**). En yoğun tekil rota **POWL → BALB**.
- **Berkeley → SF en iyi saat:** Ortalama doluluğun en düşük olduğu saat **04:00**
  (koltuk garantisi); zirve **08–09**. Akşam işten dönüşte en rahat saat ~**19:00**.

> **Heatmap (gün × saat) yorumu:** Talep iki eksende yoğunlaşıyor — *ne zaman*
> (hafta-içi sabah 08–09 ve akşam 17–18 rush hour bantları kor gibi yanıyor) ve
> *nerede* (birkaç merkez istasyon). Bu, modelde Hour/Origin/Destination/Period
> feature'larının neden önemli çıktığını da açıklıyor.

## Feature Engineering

`src/features/build_features.py`: zaman (Hour/DayOfWeek/Month/IsWeekend/Period) +
tatil (CA, IsHoliday) + mesafe (haversine dist_km) feature'ları, **self-trip drop**
(analiz 3.5), **lag/rolling** (leakage-safe, OD-bazlı) ve LightGBM için **category
dtype** (cat.codes ordinal fix, analiz 3.3).

> Not: Tam veride (~13.3M) groupby/rolling işlemi biraz sürebilir.

In [ ]:
from src.features.build_features import build_features

df = build_features(df, cfg)

print("shape:", df.shape)
lag_cols = [c for c in df.columns if "lag" in c or "roll" in c]
print("lag/rolling kolonlari:", lag_cols)
print("self-trip kalan:", int((df["Origin"] == df["Destination"]).sum()))
print("IsHoliday orani:", round(df["IsHoliday"].mean(), 4))
df.head(3)

## Model Eğitimi — Temporal Split + LightGBM (GPU) + Kayıt

Bu hücre modeli **gerçekten eğitir**: 2016 train / 2017 test (temporal split,
leakage yok), TimeSeriesSplit CV (varyans), LightGBM categorical_feature + GPU,
early stopping; sonra `models/bart_lgb_final.txt`'e kaydeder.

> Süre ~65-70 dk (CV dahil). Eğitmeden devam etmek istersen aşağıdaki **(Bilgi)**
> notundaki 'modeli yükle' yolunu kullan.

In [ ]:
from src.models.train import train_model, save_model

cfg["model"]["device"] = "gpu"   # Kaggle GPU
model, results = train_model(df, cfg)

m = results['metrics']
print("device       :", results["device"])
print("train / test :", f"{results['n_train']:,} / {results['n_test']:,}")
print("best_iter    :", results["best_iteration"])
print("holdout MAE  :", round(m["mae"], 4))
print("holdout RMSE :", round(m["rmse"], 4))
print("holdout R2   :", round(m["r2"], 4))
if "cv" in results:
    cv = results["cv"]
    print("CV MAE       :", round(cv["mean_mae"], 4), "+/-", round(cv["std_mae"], 4))

save_model(model, cfg)

### (Bilgi) Eğitim yerine modeli YÜKLEMEK istersen

Model repoda kayıtlı (`models/bart_lgb_final.txt`, clone getirir). Yukarıdaki 70 dk'lık
eğitimi tekrarlamadan devam etmek için, aşağıyı bir **kod** hücresine çevir:

```python
import os, lightgbm as lgb
from src.config import PROJECT_ROOT
model_path = os.path.join(str(PROJECT_ROOT), "models", cfg["files"]["final_model"])
model = lgb.Booster(model_file=model_path)
print("Model yuklendi:", model_path)
```

## Değerlendirme + Feature Importance (yorumlu)

Yüklenen modeli 2017 holdout'ta değerlendiriyoruz, feature importance çıkarıp
**yorumluyoruz** (analiz 3.7: 'grafik vardı, yorum yoktu' eksikliği).

### Bulgular (tam veri)
- **`Throughput_lag_1` tek başına gain'in ~%63'ü** → bir önceki saatin yolcu sayısı,
  gelecek saati belirleyen en güçlü sinyal. Talep güçlü **otokorelasyonlu**.
- **Lag + rolling birlikte ~%69** → zaman-serisi feature'ları (senior eklememiz)
  modelin belkemiği.
- Kalan: **Hour (~%9), Origin (~%7), Destination (~%6), Period (~%5)** → 'ne zaman +
  nerede' yapısı; EDA'daki yoğun saat (rush hour) ve yoğun istasyon (EMBR/MONT)
  bulgularıyla tutarlı.
- `dist_km` ~%2, gün/ay/tatil minik → mesafe sabit OD yapısında zaten Origin/Destination'a gömülü.
- **Senior not:** lag_1 baskınlığı modelin 'persistence + düzeltme' karakterini gösterir;
  lag olmayan cold-start tahmininde Hour/Origin/Destination öne çıkardı.

In [ ]:
# Hizli degerlendirme: holdout metrikleri egitimden ZATEN hazir (results) ->
# 3.2M satiri yeniden predict ETME. Feature importance modelden aninda okunur.
from src.models.evaluate import (feature_importance_df, plot_feature_importance,
                                 interpret_importance)
from src.config import get_paths

m = results["metrics"]
print("Holdout 2017 -> MAE:", round(m["mae"], 4),
      "| RMSE:", round(m["rmse"], 4), "| R2:", round(m["r2"], 4),
      "| n_test:", f"{results['n_test']:,}")

imp = feature_importance_df(model)
print()
print(imp[["feature", "gain_pct", "split"]].to_string(index=False))
print()
print("YORUM:", interpret_importance(imp))

figs = get_paths(cfg)["figures_dir"]
plot_feature_importance(imp, figs)
print("feature importance grafigi kaydedildi:", figs)

In [ ]:
# (Opsiyonel) Residual + predicted-vs-actual grafigi.
# TEK predict gerektirir (~birkac dk); portfolio gorseli icin uretiyoruz.
from src.models.evaluate import evaluate_holdout, plot_residuals

_metrics, y_true, y_pred = evaluate_holdout(model, df, cfg)
plot_residuals(y_true, y_pred, figs)
print("residual grafigi kaydedildi:", figs)

In [ ]:
from IPython.display import Image, display
import os
figs = str(get_paths(cfg)["figures_dir"])
for _name in ["feature_importance.png", "residuals.png"]:
    _p = os.path.join(figs, _name)
    if os.path.exists(_p):
        display(Image(filename=_p))

## Sonuç (Conclusion)

İki yıllık (~13M yolculuk) BART verisiyle, herhangi iki istasyon arasında saatlik
yolcu sayısını tahmin eden bir **LightGBM** modeli kurduk.

**Sonuçlar (tam veri, 2017 holdout):** MAE ≈ **3.17**, RMSE ≈ **8.49**, R² ≈ **0.937**.
TimeSeriesSplit CV: MAE **3.18 ± 0.09** (stabil, düşük varyans). İki bağımsız Kaggle
eğitimi neredeyse bire bir → **tekrarlanabilir (reproducible)**.

**Ne öğrendik:**
- Talep güçlü **zaman-otokorelasyonlu (time-autocorrelated)**: `Throughput_lag_1`
  tek başına gain'in ~%63'ü; lag + rolling birlikte ~%69. Yakın geçmiş, yakın geleceği
  belirliyor.
- Kalan sinyal **ne zaman + nerede**: Hour / Origin / Destination / Period — EDA'daki
  rush-hour ve merkez-istasyon bulgularıyla tutarlı.

**Metodolojik kazanımlar (analiz düzeltmeleri):** temporal split (leakage yok),
category dtype (ordinal hatası yok), TimeSeriesSplit CV ile varyans ölçümü, kaydedilmiş
tekrarlanabilir model, feature importance + **yazılı yorum**.

**Sınırlar & sonraki adımlar:** 2017 test penceresi yalnızca ~4 ay (Oca–May); özel
etkinlik / hava durumu verisi eklenebilir; lag'sız **cold-start** senaryosu için ayrı
bir model değerlendirilebilir.